In [69]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import math
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os
import pandas as pd
import numpy as np
from datetime import datetime
import re
import warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=UserWarning, message='Parsing.*in DD/MM/YYYY format')
# warnings.filterwarnings('ignore')  # Suppress all warnings

In [70]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None) # Show full column content

### LOAD CLEAN DATA

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent 
CLEAN_FILE = ROOT / "data" / "cleaned_EMI_dataset.csv"
OUTPUT_DIR = ROOT / "data"

In [72]:
print(f"📥 Loading cleaned dataset: {CLEAN_FILE}")
df = pd.read_csv(CLEAN_FILE)
print(f"Initial shape: {df.shape}")


📥 Loading cleaned dataset: c:\Users\malay chand\Desktop\project\guvi_project\emi\EMIPredict_AI\data\cleaned_EMI_dataset.csv
Initial shape: (404800, 27)


###  DERIVED FEATURES

In [73]:
df["debt_to_income"] = df["current_emi_amount"] / df["monthly_salary"].replace(0, np.nan)

expense_cols = [
    "school_fees", "college_fees", "travel_expenses", "groceries_utilities",
    "other_monthly_expenses", "monthly_rent"
]
df["total_monthly_expenses"] = df[expense_cols].sum(axis=1)
df["expense_to_income"] = df["total_monthly_expenses"] / df["monthly_salary"].replace(0, np.nan)
df["monthly_disposable"] = (
    df["monthly_salary"] - df["total_monthly_expenses"] - df["current_emi_amount"]
)
df["instalment_if_approved"] = df["requested_amount"] / df["requested_tenure"].replace(0, np.nan)
df["affordability_ratio"] = df["monthly_disposable"] / df["instalment_if_approved"].replace(0, np.nan)
df["employment_stability"] = df["years_of_employment"] / df["age"].replace(0, np.nan)
df["loan_to_income_ratio"] = df["requested_amount"] / df["monthly_salary"].replace(0, np.nan)
df["dependents_ratio"] = df["dependents"] / df["family_size"].replace(0, np.nan)

In [74]:
# Clean infinities
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [75]:
df.shape

(404800, 36)

In [76]:
# Get all columns by data type
print("=== Data Types ===")
print(df.dtypes)


=== Data Types ===
age                         int64
gender                     object
marital_status             object
education                  object
monthly_salary            float64
employment_type            object
years_of_employment       float64
company_type               object
house_type                 object
monthly_rent              float64
family_size                 int64
dependents                  int64
school_fees               float64
college_fees              float64
travel_expenses           float64
groceries_utilities       float64
other_monthly_expenses    float64
existing_loans             object
current_emi_amount        float64
credit_score              float64
bank_balance              float64
emergency_fund            float64
emi_scenario               object
requested_amount          float64
requested_tenure          float64
emi_eligibility            object
max_monthly_emi           float64
debt_to_income            float64
total_monthly_expenses    flo

In [77]:
# Categorical columns (object, category, bool)
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print(f"\nCategorical columns: {categorical_cols}")


Categorical columns: ['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'emi_scenario', 'emi_eligibility']


In [78]:
# Numerical columns (all numeric types)
numerical_cols = df.select_dtypes(include=['int64', 'int32', 'float64', 'float32']).columns.tolist()
print(f"Numerical columns: {numerical_cols}")

Numerical columns: ['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure', 'max_monthly_emi', 'debt_to_income', 'total_monthly_expenses', 'expense_to_income', 'monthly_disposable', 'instalment_if_approved', 'affordability_ratio', 'employment_stability', 'loan_to_income_ratio', 'dependents_ratio']


In [79]:
df['emi_scenario']

0               Personal Loan EMI
1         E-commerce Shopping EMI
2                   Education EMI
3                     Vehicle EMI
4             Home Appliances EMI
                   ...           
404795          Personal Loan EMI
404796          Personal Loan EMI
404797        Home Appliances EMI
404798        Home Appliances EMI
404799        Home Appliances EMI
Name: emi_scenario, Length: 404800, dtype: object

In [80]:
df.head()

,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,family_size,dependents,school_fees,college_fees,travel_expenses,groceries_utilities,other_monthly_expenses,existing_loans,current_emi_amount,credit_score,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi,debt_to_income,total_monthly_expenses,expense_to_income,monthly_disposable,instalment_if_approved,affordability_ratio,employment_stability,loan_to_income_ratio,dependents_ratio
0,38,Female,Married,Professional,11.321777,Private,0.9,Mid-Size,Rented,9.903538,3,2,0.0,0.00000,7200.0,19500.0,13200.0,Yes,9.903538,660.0,303200.0,11.159118,Personal Loan EMI,13.652993,15.0,Not_Eligible,500.0,0.874733,39909.903538,3525.056473,-39908.485298,0.910200,-43845.865007,0.023684,1.205905,0.666667
1,38,Female,Married,Graduate,9.975855,Private,7.0,Mnc,Family,0.000000,2,1,5100.0,0.00000,1400.0,5400.0,3500.0,Yes,8.318986,714.0,92500.0,10.199919,E-commerce Shopping EMI,11.759793,19.0,Not_Eligible,700.0,0.833912,15400.000000,1543.727372,-15398.343131,0.618936,-24878.712632,0.184211,1.178826,0.500000
2,38,Male,Married,Professional,11.363276,Private,5.8,Startup,Own,0.000000,4,3,0.0,0.00000,10200.0,19400.0,6000.0,No,0.000000,650.0,667650.0,12.497252,Education EMI,12.631344,16.0,Eligible,27775.0,0.000000,35600.000000,3132.899266,-35588.636724,0.789459,-45079.779587,0.152632,1.111593,0.750000
3,58,Female,Married,High School,11.109473,Private,2.2,Mid-Size,Own,0.000000,5,4,11400.0,0.00000,6200.0,11900.0,7900.0,No,0.000000,685.0,440900.0,12.090106,Vehicle EMI,12.624786,77.5,Eligible,16170.0,0.000000,37400.000000,3366.496223,-37388.890527,0.162900,-229519.847218,0.037931,1.136398,0.800000
4,48,Female,Married,Professional,10.956073,Private,3.4,Mid-Size,Family,0.000000,4,3,9400.0,9.69591,3600.0,16200.0,8100.0,No,0.000000,770.0,97300.0,10.247113,Home Appliances EMI,12.437188,7.0,Not_Eligible,500.0,0.000000,37309.695910,3405.389386,-37298.739836,1.776741,-20992.781634,0.070833,1.135187,0.750000


In [ ]:
df['emi_eligibility'].unique() 

array(['Not_Eligible', 'Eligible', 'High_Risk'], dtype=object)

In [ ]:
df['max_monthly_emi'].unique()

array([  500. ,   700. , 27775. , ..., 38065.5, 49860. , 23730. ],
      shape=(15383,))